# 🚜 YOLOv8 지게차·작업자 탐지: 사전학습 모델 vs 내 데이터로 학습한 모델 비교

> 제조 데이터 객체 탐지 실습 프로젝트 · Google Colab (T4 GPU) 기준
> 참고 코드: `3_YOLOv8_기반_제조_데이터_객체_탐지_실습 (프로젝트)` 노트북

## 0. 프로젝트 개요

### 문제 정의
산업 현장에서는 지게차가 계속 움직이고, 적재물·기둥 때문에 현장 관리자에게 **사각지대**가 생긴다. 사각에서 지게차와 작업자가 가까워지면 사고로 이어질 수 있다.

### 최종 목표 (장기)
카메라 영상에서 **지게차와 작업자를 실시간으로 탐지**하고, 둘 사이 거리가 임계치보다 가까워지면 **즉시 안내 방송**을 하는 시스템.

### 이번 과제의 목표
1. 참고 노트북에서 YOLOv8 실행 코드를 가져와 **사전학습(COCO) YOLOv8** 을 그대로 돌려 본다. → *Before*
2. 지게차·작업자 데이터셋으로 YOLOv8을 **추가 학습(fine-tuning)** 한다. → *After*
3. 같은 test 이미지, 같은 평가 방식으로 **Before vs After 성능을 비교하고 그래프로 시각화**한다.
4. 지게차·사람이 **일부만 보이는 상황**(화면 가장자리에서 잘림, 기둥 등에 가려짐)을 인위적으로 만들어 탐지 성능이 얼마나 유지되는지 검증하고, 이런 이미지를 섞어 재학습했을 때 개선되는지 확인한다.
5. (확장) 탐지 결과로 거리 기반 경고 로직을 간단히 시연한다.

### 가설
사전학습 YOLOv8은 COCO의 80개 클래스로 학습되어 `person` 은 잘 찾지만 **`forklift` 클래스가 없다.** 그래서 지게차는 거의 탐지하지 못할 것이고, 내 데이터로 학습하면 지게차 탐지 성능이 크게 오를 것이다.

> ✍️ 표시가 있는 셀은 **실행 결과를 보고 내 생각을 직접 적는 칸**이다. (채점 기준: 작업자 본인 의견 기록)

## 1. 환경 설정
Colab 메뉴 **Runtime(런타임) → Change runtime type(런타임 유형 변경) → T4 GPU** 선택 후 실행한다.

> 📝 이전 실습에서 겪은 문제: Colab 런타임이 초기화되면 업로드한 데이터셋이 사라져 `data.yaml does not exist` 오류가 났다.
> → 이번에는 **데이터는 매번 코드로 다시 받고**, **학습 결과는 Google Drive에 저장**하도록 구성했다.

In [ ]:
# GPU가 잡혀 있는지 확인 (Tesla T4가 보이면 정상)
!nvidia-smi
# YOLOv8을 제공하는 ultralytics 라이브러리 설치 (-q: 설치 로그 간단히)
!pip install -q ultralytics

In [ ]:
# ── 라이브러리 불러오기 ──
import os, shutil, random, time, json   # 파일 이동/삭제, 랜덤, 시간 측정, JSON 저장
from pathlib import Path                 # 경로를 문자열 대신 객체로 다루기 (/ 연산자로 경로 연결)

import numpy as np                 # 숫자 배열 계산 (박스 좌표, IoU 등)
import pandas as pd                # 결과를 표(DataFrame)로 정리
import cv2                         # 이미지 읽기/쓰기/그리기 (OpenCV, BGR 순서 사용)
import yaml                        # data.yaml 읽고 쓰기
import matplotlib.pyplot as plt    # 그래프 시각화
import torch                       # GPU 사용 가능 여부 확인
import ultralytics
from ultralytics import YOLO       # YOLOv8 모델 클래스
from IPython.display import display  # 노트북에서 표를 예쁘게 출력

# 설치된 버전과 GPU 사용 가능 여부 출력
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())

# 랜덤 시드 고정 → 다시 실행해도 같은 결과가 나오도록 (실험 재현성)
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Colab 여부 확인 → Colab이면 Drive에 결과 저장 (런타임 초기화 대비)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = Path('/content/drive/MyDrive/forklift_project')
    DATA_DIR = Path('/content/datasets/forklift_person')   # 데이터는 빠른 로컬 디스크에
else:
    WORK_DIR = Path.cwd() / 'forklift_project'
    DATA_DIR = WORK_DIR / 'datasets' / 'forklift_person'

PROJECT_DIR = WORK_DIR / 'runs'      # 학습 결과(가중치, 로그)
FIG_DIR = WORK_DIR / 'figures'       # 그래프 이미지 (README 작성 시 활용)
for d in [DATA_DIR, PROJECT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)   # 폴더가 없으면 생성 (이미 있으면 그냥 넘어감)

# ── 실험 설정값 (여기만 바꾸면 전체 실험 조건이 바뀜) ──
# GPU가 있으면 0번 GPU, 없으면 CPU 사용
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
IMGSZ, BATCH = 640, 16
EPOCHS = 50          # T4 + 이미지 약 400장 기준 10~15분. 시간이 없으면 30으로 줄이기
CONF_TH = 0.25       # Precision/Recall 계산과 시각화에 사용할 confidence 기준
IOU_TH = 0.5         # 예측 박스가 정답으로 인정되는 IoU 기준 (mAP50과 동일)

def savefig(name):
    # 현재 그래프를 figures 폴더에 png로 저장 (보고서/README에 첨부용)
    plt.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')

print('WORK_DIR:', WORK_DIR, '| DATA_DIR:', DATA_DIR, '| DEVICE:', DEVICE)

## 2. 사전학습 YOLOv8 동작 확인 (참고 노트북에서 가져온 코드)
참고 노트북의 앞부분(`YOLO('yolov8n.pt')` → `predict` → `results[0].boxes`)을 그대로 가져와 모델이 정상 동작하는지 확인한다.

In [ ]:
# 참고 노트북 코드: COCO로 사전학습된 YOLOv8n 불러와서 예시 이미지 추론
pretrained = YOLO('yolov8n.pt')

example_url = 'https://ultralytics.com/images/bus.jpg'
# conf=0.25: 신뢰도 25% 이상인 박스만 결과로 남김 / verbose=False: 로그 출력 생략
results = pretrained.predict(example_url, conf=0.25, verbose=False)

# results[0].plot(): 예측 박스가 그려진 이미지(BGR) → matplotlib용 RGB로 변환해서 표시
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
plt.title('Pretrained YOLOv8n (COCO) - example'); plt.axis('off'); plt.show()

# boxes.data 한 줄 = [x1, y1, x2, y2, conf, cls]  (참고 노트북에서 확인한 구조)
print(results[0].boxes.data[:3])

In [ ]:
# COCO 클래스 목록 확인: 'person'은 있지만 'forklift'는 없다
COCO_NAMES = pretrained.names          # {0: 'person', 1: 'bicycle', ...}
# 우리 과제에 필요한 클래스가 COCO에 있는지 확인
print('클래스 수:', len(COCO_NAMES))
print('person 있음? ', 'person' in COCO_NAMES.values())
print('forklift 있음?', any('fork' in v for v in COCO_NAMES.values()))
print('지게차와 비슷한 차량 클래스:', {k: v for k, v in COCO_NAMES.items() if v in ['car', 'truck', 'bus']})

### ✍️ 관찰
- 사전학습 모델은 사람을 ___ 하게 탐지했다.
- COCO에는 `forklift` 가 없으므로, 사전학습 모델로 지게차를 잡으려면 가장 비슷한 `truck` 클래스로 대신 판단할 수밖에 없다. → 이 한계가 **추가 학습이 필요한 이유**다.

## 3. 데이터 준비

### 사용 데이터셋
- **Forklift-Person-Detector** (Roboflow Universe, 작성자 luzuko) — https://universe.roboflow.com/luzuko/forklift-person-detector
- 클래스 2개(`forklift`, `person`), 약 420장, 라이선스 CC BY 4.0
- 직접 모은 이미지가 있다면 Roboflow에서 라벨링 후 같은 YOLOv8 형식으로 내보내 사용한다.

### 불러오는 방법 (`DATA_SOURCE`)
| 값 | 방법 |
|---|---|
| `'roboflow'` | Roboflow API로 다운로드. API 키는 코드에 쓰지 않고 Colab 왼쪽 🔑 **Secrets(보안 비밀)** 에 `ROBOFLOW_API_KEY` 이름으로 등록 |
| `'zip'` | Roboflow에서 **Download Dataset → YOLOv8 → Download zip to computer** 로 받은 zip을 Drive에 올려두고 사용 |
| `'ready'` | 이미 `DATA_DIR` 에 풀려 있는 경우 |

> 📝 참고 노트북에서는 API 키를 코드에 그대로 적었는데, 노트북을 공개 링크로 제출하면 키가 노출되므로 Secrets를 쓰도록 바꿨다.

In [ ]:
# ── 데이터셋 가져오기: 방법 3가지 중 하나 선택 ──
DATA_SOURCE = 'roboflow'   # 'roboflow' | 'zip' | 'ready'

RF_WORKSPACE, RF_PROJECT = 'luzuko', 'forklift-person-detector'
RF_VERSION = 1             # Roboflow 페이지에서 버전 번호 확인 후 수정
ZIP_PATH = '/content/drive/MyDrive/datasets/forklift_person.zip'

if DATA_SOURCE == 'roboflow':
    # roboflow 라이브러리 설치 (파이썬 코드 안에서 pip 실행)
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'], check=True)
    from roboflow import Roboflow
    # API 키 가져오기: Colab Secrets에 저장된 값 사용 (없으면 입력창에서 직접 입력, 화면에 표시 안 됨)
    try:
        from google.colab import userdata
        api_key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        import getpass
        api_key = getpass.getpass('Roboflow API key: ')
    # 워크스페이스 → 프로젝트 → 버전 순서로 찾아서 YOLOv8 형식으로 DATA_DIR에 다운로드
    rf = Roboflow(api_key=api_key)
    rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
        'yolov8', location=str(DATA_DIR), overwrite=True)
elif DATA_SOURCE == 'zip':
    # Drive에 올려둔 zip 파일을 DATA_DIR에 압축 해제
    shutil.unpack_archive(ZIP_PATH, DATA_DIR)

# 받은 폴더 구성 확인 (train, valid, test, data.yaml 등이 보여야 정상)
print(sorted(p.name for p in DATA_DIR.iterdir()))

### data.yaml 경로 정리
Roboflow의 `data.yaml` 은 `../train/images` 같은 상대 경로를 써서 환경에 따라 경로 오류가 난다.
참고 노트북에서 했던 것처럼 **절대 경로로 바꾼 yaml** 을 새로 만든다. `valid`/`test` 폴더가 없는 데이터셋이면 train 일부를 떼어 만든다.

In [ ]:
# 이미지로 인정할 확장자
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(d):
    # 폴더 안의 이미지 파일 경로를 정렬된 리스트로 반환
    return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in IMG_EXTS)

def label_path(img_path):
    # YOLO 형식: .../images/a.jpg 의 라벨은 .../labels/a.txt
    img_path = Path(img_path)
    return img_path.parent.parent / 'labels' / (img_path.stem + '.txt')

# ── ① 원본 data.yaml 찾기 (zip 구조에 따라 하위 폴더에 있을 수 있어 rglob으로 전체 검색) ──
yaml_files = sorted(DATA_DIR.rglob('data.yaml'))
assert yaml_files, f'{DATA_DIR} 에서 data.yaml을 찾지 못했습니다. 데이터 다운로드 셀을 확인하세요.'
DATA_ROOT = yaml_files[0].parent
raw_cfg = yaml.safe_load(open(yaml_files[0]))
# 클래스 이름 목록 (yaml에 리스트 또는 {번호: 이름} 딕셔너리로 적혀 있을 수 있어 둘 다 처리)
names = raw_cfg['names']
CLASS_NAMES = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)

# ── ② train / val / test 이미지 폴더 찾기 ──
def find_split(*cands):
    # 후보 이름 중 이미지가 실제로 들어 있는 폴더를 반환 (Roboflow는 val을 'valid'로 씀)
    for c in cands:
        p = DATA_ROOT / c / 'images'
        if p.exists() and list_images(p):
            return p
    return None

split_dirs = {'train': find_split('train'), 'val': find_split('valid', 'val'), 'test': find_split('test')}
assert split_dirs['train'] is not None, 'train/images 폴더가 없습니다.'

def carve_split(new_split, ratio):
    # train 일부를 새 split으로 이동 (val/test가 없는 데이터셋 대비)
    imgs = list_images(split_dirs['train'])
    chosen = random.Random(SEED).sample(imgs, max(1, int(len(imgs) * ratio)))
    (DATA_ROOT / new_split / 'images').mkdir(parents=True, exist_ok=True)
    (DATA_ROOT / new_split / 'labels').mkdir(parents=True, exist_ok=True)
    for img in chosen:
        lbl = label_path(img)
        shutil.move(str(img), DATA_ROOT / new_split / 'images' / img.name)
        if lbl.exists():
            shutil.move(str(lbl), DATA_ROOT / new_split / 'labels' / lbl.name)
    print(f'[{new_split}] train에서 {len(chosen)}장 분리')
    return DATA_ROOT / new_split / 'images'

# val이 없으면 train의 15%, test가 없으면 10%를 떼어서 만든다
if split_dirs['val'] is None:
    split_dirs['val'] = carve_split('valid', 0.15)
if split_dirs['test'] is None:
    split_dirs['test'] = carve_split('test', 0.10)

# ── ③ 절대 경로로 된 새 yaml 저장 → 학습/평가 모두 이 파일을 사용 (경로 오류 방지) ──
YAML_PATH = DATA_ROOT / 'data_abs.yaml'
yaml.safe_dump({'path': str(DATA_ROOT), 'train': str(split_dirs['train']), 'val': str(split_dirs['val']),
                'test': str(split_dirs['test']), 'nc': len(CLASS_NAMES), 'names': CLASS_NAMES},
               open(YAML_PATH, 'w'), allow_unicode=True, sort_keys=False)

# ── ④ 지게차/사람 클래스 번호 자동 찾기 (데이터셋마다 번호 순서가 달라서) ──
def find_class_id(keywords):
    # 클래스 이름에 키워드가 들어간 첫 번째 클래스 번호 반환, 없으면 None
    return next((i for i, n in enumerate(CLASS_NAMES) if any(k in n.lower() for k in keywords)), None)

FORKLIFT_ID = find_class_id(['fork'])
PERSON_ID = find_class_id(['person', 'people', 'worker', 'human', 'pedestrian'])
assert FORKLIFT_ID is not None and PERSON_ID is not None, \
    f'클래스 이름에서 지게차/사람을 찾지 못했습니다: {CLASS_NAMES} → FORKLIFT_ID, PERSON_ID 직접 지정'
EVAL_CLASSES = [FORKLIFT_ID, PERSON_ID]   # 평가할 클래스 목록

print('클래스:', CLASS_NAMES, '| forklift =', FORKLIFT_ID, '| person =', PERSON_ID)
for k, v in split_dirs.items():
    print(f'{k:5s}: {len(list_images(v)):4d}장')

## 4. 데이터 살펴보기 (EDA)
- 클래스별 객체 수가 균형적인지
- 지게차와 사람이 **한 이미지에 같이 있는 경우**가 충분한지 (거리 경고에 꼭 필요한 상황)

In [ ]:
def read_labels(img_path, W=None, H=None):
    # 라벨 파일 → [(cls, x1, y1, x2, y2)]. W, H를 주면 픽셀 좌표, 없으면 0~1 상대 좌표
    out, lp = [], label_path(img_path)
    if not lp.exists():
        return out
    for ln in lp.read_text().splitlines():
        parts = ln.split()
        if len(parts) < 5:
            continue
        c, v = int(parts[0]), list(map(float, parts[1:]))
        if len(v) == 4:
            x, y, w, h = v
            x1, y1, x2, y2 = x - w / 2, y - h / 2, x + w / 2, y + h / 2
        else:   # 폴리곤 라벨이면 감싸는 박스로 변환
            x1, x2, y1, y2 = min(v[0::2]), max(v[0::2]), min(v[1::2]), max(v[1::2])
        if W is not None:
            x1, x2, y1, y2 = x1 * W, x2 * W, y1 * H, y2 * H
        out.append((c, x1, y1, x2, y2))
    return out

# ── 이미지마다 라벨을 읽어 클래스별 개수를 표로 정리 ──
rows = []
for split, d in split_dirs.items():
    for p in list_images(d):
        labels = read_labels(p)
        cls = [l[0] for l in labels]
        areas = [(l[3] - l[1]) * (l[4] - l[2]) for l in labels]
        rows.append(dict(split=split, path=str(p), n_forklift=cls.count(FORKLIFT_ID), n_person=cls.count(PERSON_ID),
                         min_area=min(areas) if areas else np.nan))
images_df = pd.DataFrame(rows)
# 지게차와 사람이 한 이미지에 함께 있는지 (거리 경고에 쓸 수 있는 이미지)
images_df['both'] = (images_df['n_forklift'] > 0) & (images_df['n_person'] > 0)

# split별 요약: 이미지 수, 클래스별 박스 수, 둘 다 있는 이미지 수
summary = images_df.groupby('split').agg(images=('path', 'count'), forklift_boxes=('n_forklift', 'sum'),
                                         person_boxes=('n_person', 'sum'), images_with_both=('both', 'sum'))
display(summary.reindex(['train', 'val', 'test']))

# 클래스별 객체 수 막대그래프 → 클래스 불균형 확인
summary[['forklift_boxes', 'person_boxes']].reindex(['train', 'val', 'test']).plot(kind='bar', rot=0, figsize=(7, 4))
plt.title('Instances per class / split'); plt.grid(alpha=0.3); savefig('eda_instances'); plt.show()

In [ ]:
# 클래스별 박스 색 (OpenCV는 BGR 순서: 지게차=주황, 사람=파랑)
COLORS = {FORKLIFT_ID: (0, 165, 255), PERSON_ID: (255, 80, 0)}

def draw_gt(img_path):
    # 정답(Ground Truth) 박스를 이미지에 그려서 RGB로 반환
    img = cv2.imread(str(img_path)); H, W = img.shape[:2]
    for c, x1, y1, x2, y2 in read_labels(img_path, W, H):
        color = COLORS.get(c, (0, 255, 0))
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(img, CLASS_NAMES[c], (int(x1), max(int(y1) - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_grid(images, titles=None, ncols=4, size=4, name=None):
    # 여러 이미지를 격자로 한 번에 보여주고, name이 있으면 png로 저장
    nrows = int(np.ceil(len(images) / ncols))
    plt.figure(figsize=(size * ncols, size * nrows))
    for i, im in enumerate(images):
        plt.subplot(nrows, ncols, i + 1); plt.imshow(im); plt.axis('off')
        if titles: plt.title(titles[i], fontsize=10)
    plt.tight_layout()
    if name: savefig(name)
    plt.show()

# train 이미지 8장을 무작위로 뽑아 정답 박스 확인 (라벨이 제대로 되어 있는지 눈으로 점검)
train_df = images_df[images_df['split'] == 'train']
sample = train_df.sample(min(8, len(train_df)), random_state=SEED)
show_grid([draw_gt(p) for p in sample['path']], name='eda_gt_samples')

### ✍️ EDA 관찰
- 클래스 균형: 
- 지게차와 사람이 함께 있는 이미지 수: 
- 이미지 환경(실내/실외, 밝기, 카메라 각도)이 실제 현장과 비슷한가?: 

## 5. 공정한 비교를 위한 평가 함수
Before/After를 **똑같은 test 이미지, 똑같은 기준**으로 평가해야 비교가 의미 있다.

- 사전학습 모델의 클래스 번호(COCO)와 내 데이터셋 클래스 번호가 다르기 때문에 Ultralytics의 `model.val()` 을 그대로 쓸 수 없다.
- 그래서 **예측 박스와 정답 박스의 IoU를 직접 계산**해 클래스별 Precision, Recall, AP50을 구하는 함수를 만들었다.
- 사전학습 모델 클래스 매핑: COCO `person` → 우리 `person`, COCO `truck` → 우리 `forklift` (지게차 클래스가 없어 가장 비슷한 차량으로 대체)

| 지표 | 의미 |
|---|---|
| Precision | 모델이 찾았다고 한 것 중 실제로 맞은 비율 (오탐이 적을수록 높음) |
| Recall | 실제 객체 중 모델이 찾아낸 비율 (**놓치면 사고로 이어지므로 가장 중요**) |
| AP50 | IoU 0.5 기준, confidence를 바꿔가며 그린 PR 곡선의 넓이 (종합 성능) |

In [ ]:
def iou_matrix(a, b):
    # a: (N,4), b: (M,4) xyxy → (N,M) IoU
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def average_precision(recall, precision):
    # 모든 recall 지점에서 보간한 PR 곡선의 넓이 (VOC/COCO 방식)
    mrec = np.concatenate(([0.0], recall, [1.0])); mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

def predict_mapped(model, img_path, class_map, imgsz=IMGSZ):
    # 모델 예측 → [x1,y1,x2,y2,conf,우리 데이터셋 클래스] 배열. class_map에 없는 클래스는 버림
    r = model.predict(str(img_path), imgsz=imgsz, conf=0.001, device=DEVICE, verbose=False,
                      classes=list(class_map.keys()))[0]
    d = r.boxes.data.cpu().numpy()
    if len(d) == 0:
        return np.zeros((0, 6))
    d[:, 5] = [class_map[int(c)] for c in d[:, 5]]
    return d

def evaluate(model, class_map, name, img_dir=None, imgsz=IMGSZ):
    # 모델 하나를 test 이미지 전체에 적용해 클래스별 P / R / AP50 계산
    img_dir = img_dir or split_dirs['test']
    # 클래스별로 [예측 신뢰도 목록, 정답 여부 목록, 정답 박스 총 개수]를 모은다
    stats = {c: {'conf': [], 'tp': [], 'n_gt': 0} for c in EVAL_CLASSES}
    t0 = time.perf_counter(); n_img = 0
    for p in list_images(img_dir):
        # 정답 박스(픽셀 좌표)와 예측 박스 준비
        H, W = cv2.imread(str(p)).shape[:2]
        gt = np.array(read_labels(p, W, H)).reshape(-1, 5)
        pred = predict_mapped(model, p, class_map, imgsz); n_img += 1
        for c in EVAL_CLASSES:
            # 이 클래스의 정답 박스(g)와 예측 박스(pr, 신뢰도 높은 순 정렬)
            g = gt[gt[:, 0] == c][:, 1:]
            pr = pred[pred[:, 5] == c]; pr = pr[np.argsort(-pr[:, 4])]
            stats[c]['n_gt'] += len(g)
            ious = iou_matrix(pr[:, :4], g); matched = np.zeros(len(g), bool)
            for i in range(len(pr)):   # confidence 높은 예측부터 가장 많이 겹치는 정답과 1:1 매칭
                j = int(np.argmax(ious[i])) if len(g) else -1
                hit = j >= 0 and ious[i, j] >= IOU_TH and not matched[j]
                if hit: matched[j] = True
                stats[c]['conf'].append(pr[i, 4]); stats[c]['tp'].append(hit)
    elapsed = time.perf_counter() - t0

    # ── 모은 결과로 지표 계산 ──
    row, curves = {'model': name}, {}
    for c in EVAL_CLASSES:
        conf, tp, n_gt = np.array(stats[c]['conf']), np.array(stats[c]['tp'], bool), stats[c]['n_gt']
        order = np.argsort(-conf); conf, tp = conf[order], tp[order]
        # 신뢰도 기준을 점점 낮추며 누적한 정답(TP)/오답(FP) 수 → 각 지점의 Recall, Precision
        ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
        rec = ctp / max(n_gt, 1); prec = ctp / np.maximum(ctp + cfp, 1)
        k = int((conf >= CONF_TH).sum())   # CONF_TH 이상인 예측만 남겼을 때
        cname = CLASS_NAMES[c]
        row[f'{cname}_AP50'] = average_precision(rec, prec) if len(conf) else 0.0
        row[f'{cname}_P'] = float(prec[k - 1]) if k else 0.0
        row[f'{cname}_R'] = float(rec[k - 1]) if k else 0.0
        row[f'{cname}_n_gt'] = n_gt
        curves[cname] = (rec, prec)
    # 클래스별 AP50의 평균 = mAP50 / 이미지 1장 처리 시간(이미지 읽기 포함)
    row['mAP50'] = float(np.mean([row[f'{CLASS_NAMES[c]}_AP50'] for c in EVAL_CLASSES]))
    row['sec_per_img'] = elapsed / max(n_img, 1)
    return row, curves

## 6. [Before] 사전학습 YOLOv8n 평가
내 데이터로 학습하기 **전**, COCO 사전학습 모델을 그대로 test 이미지에 적용한다.

In [ ]:
# COCO 클래스 번호 → 우리 데이터셋 클래스 번호
COCO_PERSON = next(k for k, v in COCO_NAMES.items() if v == 'person')
COCO_TRUCK = next(k for k, v in COCO_NAMES.items() if v == 'truck')
PRETRAINED_MAP = {COCO_PERSON: PERSON_ID, COCO_TRUCK: FORKLIFT_ID}

# 사전학습 모델로 test셋 평가 → 결과 표(before_row)와 PR 곡선 데이터(before_curves)
before_row, before_curves = evaluate(pretrained, PRETRAINED_MAP, 'Before (COCO pretrained)')
pd.Series(before_row).round(3)

### ✍️ Before 결과 해석
- 사람(person) AP50 = ___ : 사전학습만으로도 사람은 ___
- 지게차(forklift) AP50 = ___ : truck으로 대체했을 때 ___

## 7. 내 데이터로 YOLOv8n 추가 학습 (Fine-tuning)
- 사전학습 가중치 `yolov8n.pt` 에서 시작해 지게차·사람 데이터로 추가 학습한다. (참고 노트북의 `model.train(...)` 과 동일한 방식)
- 결과는 Drive(`PROJECT_DIR`)에 저장되고, **Colab이 중간에 끊기면 이 셀을 다시 실행해 `last.pt` 에서 이어서 학습**한다.

| 실험 조건 | 값 |
|---|---|
| 모델 | YOLOv8n (COCO 사전학습 가중치에서 시작) |
| 입력 크기 / batch | 640 / 16 |
| epochs | `EPOCHS` (기본 50) |
| 데이터 증강 | Ultralytics 기본값 (mosaic, flip, 색상 변환 등) |

In [ ]:
def completed_epochs(run_dir):
    # results.csv의 줄 수 = 지금까지 끝난 epoch 수
    csv = Path(run_dir) / 'results.csv'
    return len(pd.read_csv(csv)) if csv.exists() else 0

def train_or_resume(run_name, data_yaml, base_weights='yolov8n.pt'):
    # 학습을 실행하고 best.pt 경로를 반환 (10장 부분 노출 재학습에서도 같은 함수 재사용)
    # best.pt: val 성능이 가장 좋았던 epoch의 가중치 / last.pt: 마지막 epoch의 가중치
    run_dir = PROJECT_DIR / run_name
    best_pt, last_pt = run_dir / 'weights' / 'best.pt', run_dir / 'weights' / 'last.pt'
    t0 = time.time()
    # 경우 ① 이미 학습 완료 → 다시 학습하지 않음
    if best_pt.exists() and completed_epochs(run_dir) >= EPOCHS:
        print(f'[{run_name}] 이미 학습이 끝났습니다 → 학습 생략')
    # 경우 ② 학습 도중 Colab이 끊김 → 마지막 저장 지점부터 이어서 학습
    elif last_pt.exists():
        print(f'[{run_name}] 중단된 학습 이어서 진행 ({completed_epochs(run_dir)}/{EPOCHS} epoch 완료 상태)')
        YOLO(str(last_pt)).train(resume=True)
    # 경우 ③ 처음 학습 → 사전학습 가중치에서 시작해 내 데이터로 추가 학습
    #   cache=True: 이미지를 메모리에 올려 두 번째 epoch부터 빠르게 / seed: 재현성
    else:
        shutil.rmtree(run_dir, ignore_errors=True)   # 남아 있던 불완전한 결과 삭제
        YOLO(base_weights).train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                 project=str(PROJECT_DIR), name=run_name, exist_ok=True, seed=SEED, cache=True)
    print(f'[{run_name}] 소요 시간: {(time.time() - t0) / 60:.1f}분')
    return best_pt

RUN_NAME = 'finetune_yolov8n'
run_dir = PROJECT_DIR / RUN_NAME
best_pt = train_or_resume(RUN_NAME, YAML_PATH)

In [ ]:
# 학습 곡선: loss가 줄고 mAP가 오르는지, val loss가 다시 오르는 과적합 구간이 있는지 확인
# epoch별 학습 기록 불러오기 (열 이름 앞뒤 공백 제거)
hist = pd.read_csv(run_dir / 'results.csv'); hist.columns = [c.strip() for c in hist.columns]
# 열 이름이 버전마다 조금씩 달라서, 키워드가 포함된 열을 찾아주는 함수
col = lambda key: next(c for c in hist.columns if key in c)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for key in ['box_loss', 'cls_loss']:
    axes[0].plot(hist['epoch'], hist[f'train/{key}'], label=f'train {key}')
    axes[0].plot(hist['epoch'], hist[f'val/{key}'], '--', label=f'val {key}')
axes[0].set_title('Loss')
axes[1].plot(hist['epoch'], hist[col('precision')], label='precision')
axes[1].plot(hist['epoch'], hist[col('recall')], label='recall'); axes[1].set_title('Precision / Recall (val)')
axes[2].plot(hist['epoch'], hist[col('mAP50(')], label='mAP50')
axes[2].plot(hist['epoch'], hist[col('mAP50-95')], label='mAP50-95'); axes[2].set_title('mAP (val)')
for ax in axes: ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); savefig('finetune_learning_curves'); plt.show()

### ✍️ 학습 과정 관찰
- loss 추세 / 과적합 여부: 
- 몇 epoch 즈음 성능이 수렴했는가?: 

## 8. [After] 학습한 모델 평가
Before와 **같은 평가 함수**로 평가하고, 참고용으로 Ultralytics 공식 평가(`model.val`)도 함께 확인한다.

In [ ]:
# 가장 성능이 좋았던 가중치로 모델 불러오기
finetuned = YOLO(str(best_pt))
FINETUNED_MAP = {i: i for i in range(len(CLASS_NAMES))}   # 클래스 번호가 같으므로 그대로

# Before와 똑같은 함수·이미지로 평가 → 공정한 비교
after_row, after_curves = evaluate(finetuned, FINETUNED_MAP, 'After (fine-tuned)')
display(pd.Series(after_row).round(3))

# 참고: Ultralytics 공식 test 평가 (mAP50-95, 혼동행렬 등이 PROJECT_DIR/finetune_test 에 저장됨)
official = finetuned.val(data=str(YAML_PATH), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                         project=str(PROJECT_DIR), name='finetune_test', exist_ok=True, verbose=False)
print(f'[Ultralytics] mAP50={official.box.map50:.3f}  mAP50-95={official.box.map:.3f}')

## 9. Before vs After 비교 시각화

In [ ]:
# Before/After 결과를 한 표로 합치고 CSV로 저장
compare = pd.DataFrame([before_row, after_row]).set_index('model')
compare['FPS'] = 1 / compare['sec_per_img']   # 초당 처리 이미지 수 (클수록 빠름)
compare.to_csv(WORK_DIR / 'before_after_results.csv')
show_cols = ['mAP50'] + [f'{CLASS_NAMES[c]}_{m}' for c in EVAL_CLASSES for m in ['AP50', 'P', 'R']] + ['FPS']
display(compare[show_cols].round(3))

In [ ]:
# ── 지표 3개(AP50, Precision, Recall)를 클래스별 Before/After 막대로 비교 ──
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
cls_names = [CLASS_NAMES[c] for c in EVAL_CLASSES]
x, w = np.arange(len(cls_names)), 0.35

for ax, metric in zip(axes, ['AP50', 'P', 'R']):
    # b: Before 값, a: After 값 (클래스 순서대로)
    b = [compare.iloc[0][f'{n}_{metric}'] for n in cls_names]
    a = [compare.iloc[1][f'{n}_{metric}'] for n in cls_names]
    ax.bar(x - w / 2, b, w, label='Before (pretrained)', color='gray')
    ax.bar(x + w / 2, a, w, label='After (fine-tuned)', color='tab:blue')
    # 막대 위에 수치 표시
    for i in range(len(cls_names)):
        ax.text(x[i] - w / 2, b[i] + 0.02, f'{b[i]:.2f}', ha='center')
        ax.text(x[i] + w / 2, a[i] + 0.02, f'{a[i]:.2f}', ha='center')
    ax.set_xticks(x); ax.set_xticklabels(cls_names); ax.set_ylim(0, 1.1)
    title = {'AP50': 'AP50', 'P': f'Precision (conf>={CONF_TH})', 'R': f'Recall (conf>={CONF_TH})'}[metric]
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); savefig('before_after_bars'); plt.show()

In [ ]:
# PR 곡선: 곡선이 오른쪽 위로 갈수록 좋은 모델 (넓이 = AP50)
# 클래스마다 그래프 1개, 그 안에 Before(회색)/After(파랑) 곡선을 겹쳐 그림
fig, axes = plt.subplots(1, len(cls_names), figsize=(6 * len(cls_names), 4.5))
for ax, n in zip(np.atleast_1d(axes), cls_names):
    for curves, label, color in [(before_curves, 'Before', 'gray'), (after_curves, 'After', 'tab:blue')]:
        rec, prec = curves[n]
        ax.plot(rec, prec, color=color, label=f"{label} (AP50={compare.loc[compare.index.str.startswith(label)].iloc[0][f'{n}_AP50']:.2f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05); ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR curve - {n}'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); savefig('before_after_pr_curves'); plt.show()

In [ ]:
# 같은 이미지에서 정답 / Before / After 예측 비교 (지게차+사람이 함께 있는 test 이미지 우선)
test_imgs = images_df[images_df['split'] == 'test'].sort_values('both', ascending=False)['path'].head(3)
# 이미지 1장당 [정답, Before 예측, After 예측] 3장을 한 줄로 배치
ims, titles = [], []
for p in test_imgs:
    ims.append(draw_gt(p)); titles.append('Ground Truth')
    r = pretrained.predict(p, conf=CONF_TH, classes=list(PRETRAINED_MAP), device=DEVICE, verbose=False)[0]
    ims.append(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); titles.append('Before (COCO: person/truck)')
    r = finetuned.predict(p, conf=CONF_TH, device=DEVICE, verbose=False)[0]
    ims.append(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); titles.append('After (fine-tuned)')
show_grid(ims, titles, ncols=3, size=5, name='before_after_examples')

### ✍️ Before vs After 비교 해석
| 항목 | Before | After | 변화 |
|---|---|---|---|
| forklift AP50 | | | |
| person AP50 | | | |
| person Recall | | | |

- 가설(지게차 탐지 성능이 크게 오를 것)이 맞았는가?: 
- 사람 탐지는 원래도 잘 됐는데, 학습 후 어떻게 변했는가? (현장 데이터에 맞춰져 좋아졌는지 / 비슷한지): 
- 예시 이미지에서 눈에 띈 차이: 
- 아직 틀리는 사례와 원인 추측: 

## 10. 부분 노출 실험: 일부만 보여도 탐지하는가?
사각지대 사고는 대부분 지게차나 사람이 **기둥·적재물에 가려지거나 카메라 시야 끝에 걸려 일부만 보이는 상황**에서 일어난다.
원본 이미지와 정답 라벨을 이용해 이런 상황을 **코드로 자동 생성**하고(라벨도 자동 보정), 두 단계로 검증한다.

| 변형 | 만드는 방법 | 가정한 현장 상황 | 라벨 처리 |
|---|---|---|---|
| **crop** | 지게차/사람 하나를 골라 그 객체를 가로지르는 선으로 이미지를 자름 (객체의 30~70%가 잘려나감) | 카메라 시야 가장자리에 일부만 걸림 | 잘린 범위로 박스를 줄이고, 보이는 면적이 `MIN_VISIBLE`(20%) 미만이면 제외 |
| **occlude30 / 50** | 각 지게차/사람 박스의 한쪽 면부터 30% / 50%를 회색 막대로 덮음 | 기둥·적재물에 가려짐 | 박스를 **보이는 부분**으로 줄임 |

- **1단계 (평가만)**: 변형된 test 이미지로 Before/After 모델을 평가 → 지금 모델이 부분 노출에 얼마나 약한지
- **2단계 (재학습)**: train 이미지 일부로 변형본을 만들어 원본에 **추가**하고 재학습 → 원본 test와 변형 test **모두**에서 비교 (부분 노출에 강해지면서 원래 성능은 유지되는지)

> 원본 데이터는 수정하지 않고 `partial/` 폴더에 따로 저장한다. 시드를 고정했기 때문에 다시 실행해도 같은 변형 데이터가 만들어진다.
> 한계: 규칙에 따라 자르고 덮은 인위적인 데이터라 실제로 가려진 사진과는 다르다. 객체끼리 겹쳐 있으면 한 객체의 가림막이 다른 객체를 덮어도 그 객체의 라벨은 줄어들지 않는다.

In [ ]:
# ── 부분 노출 실험 설정값 (✍️ 바꿔가며 실험해 볼 수 있는 조건들) ──
PARTIAL_DIR = DATA_DIR.parent / 'partial'   # 변형 데이터 저장 위치 (원본 데이터는 건드리지 않음)
MIN_VISIBLE = 0.2                 # 원래 박스 대비 보이는 면적이 20% 미만이면 라벨에서 제외 (사람이 봐도 판단하기 어려운 수준)
CUT_RANGE = (0.3, 0.7)            # crop: 대상 객체가 잘려나가는 비율 범위
OCCLUDE_RATIOS = [0.3, 0.5]       # occlude: 박스를 가리는 비율
OCCLUDE_COLOR = (128, 128, 128)   # 가림막 색 (회색 = 기둥/적재물을 단순하게 가정)
TRAIN_AUG_RATIO = 0.5             # 2단계: train 이미지의 50%로 변형본을 만들어 원본에 추가

def write_yolo_labels(path, boxes, W, H):
    # 픽셀 좌표 박스 [(cls, x1, y1, x2, y2)] → YOLO 형식(cls cx cy w h, 0~1 비율)으로 저장
    lines = [f'{int(c)} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}'
             for c, x1, y1, x2, y2 in boxes]
    Path(path).write_text('\n'.join(lines) + ('\n' if lines else ''))

def crop_edge(img, boxes, rng):
    # 지게차/사람 중 하나를 무작위로 골라, 그 객체를 가로지르는 선으로 이미지를 잘라 일부만 남긴다
    H, W = img.shape[:2]
    targets = [b for b in boxes if b[0] in EVAL_CLASSES]
    if not targets:                       # 대상 객체가 없는 이미지는 그대로 둠
        return img, boxes, None
    c, x1, y1, x2, y2 = targets[rng.integers(len(targets))]
    f = rng.uniform(*CUT_RANGE)           # 잘려나갈 비율
    # 네 방향 후보: 자르는 선의 위치와, 남길 영역(rx1, ry1, rx2, ry2)
    cands = {'left':   (int(x1 + f * (x2 - x1)), 0, W, H),
             'right':  (0, 0, int(x2 - f * (x2 - x1)), H),
             'top':    (0, int(y1 + f * (y2 - y1)), W, H),
             'bottom': (0, 0, W, int(y2 - f * (y2 - y1)))}
    # 남는 이미지가 원본의 40%보다 작아지는 방향은 제외 (이미지가 너무 작아지는 것 방지)
    ok = [(k, r) for k, r in cands.items() if r[2] - r[0] >= 0.4 * W and r[3] - r[1] >= 0.4 * H]
    if not ok:
        return img, boxes, None
    side, (rx1, ry1, rx2, ry2) = ok[rng.integers(len(ok))]
    new_img = img[ry1:ry2, rx1:rx2].copy()
    # 모든 박스를 새 이미지 좌표로 옮기고 잘린 범위에 맞게 자름
    new_boxes = []
    for bc, bx1, by1, bx2, by2 in boxes:
        nx1, ny1 = max(bx1, rx1) - rx1, max(by1, ry1) - ry1
        nx2, ny2 = min(bx2, rx2) - rx1, min(by2, ry2) - ry1
        if nx2 <= nx1 or ny2 <= ny1:      # 완전히 잘려나간 박스
            continue
        visible = (nx2 - nx1) * (ny2 - ny1) / max((bx2 - bx1) * (by2 - by1), 1e-9)
        if visible >= MIN_VISIBLE:        # 충분히 보이는 박스만 라벨로 남김
            new_boxes.append((bc, nx1, ny1, nx2, ny2))
    return new_img, new_boxes, f'{CLASS_NAMES[int(c)]} cut {f:.0%} ({side})'

def occlude(img, boxes, ratio, rng):
    # 지게차/사람 박스마다 무작위 한쪽 면부터 ratio만큼 회색 막대로 덮고, 라벨은 보이는 부분으로 줄인다
    img, new_boxes = img.copy(), []
    for c, x1, y1, x2, y2 in boxes:
        if c not in EVAL_CLASSES:
            new_boxes.append((c, x1, y1, x2, y2)); continue
        w, h = x2 - x1, y2 - y1
        side = ['left', 'right', 'top', 'bottom'][rng.integers(4)]
        if side == 'left':    cover, vis = (x1, y1, x1 + ratio * w, y2), (x1 + ratio * w, y1, x2, y2)
        elif side == 'right': cover, vis = (x2 - ratio * w, y1, x2, y2), (x1, y1, x2 - ratio * w, y2)
        elif side == 'top':   cover, vis = (x1, y1, x2, y1 + ratio * h), (x1, y1 + ratio * h, x2, y2)
        else:                 cover, vis = (x1, y2 - ratio * h, x2, y2), (x1, y1, x2, y2 - ratio * h)
        cv2.rectangle(img, (int(cover[0]), int(cover[1])), (int(cover[2]), int(cover[3])), OCCLUDE_COLOR, -1)
        new_boxes.append((c, *vis))
    return img, new_boxes

def make_variant(src_dir, dst_name, mode, ratio=None, sample_ratio=1.0):
    # src_dir의 이미지로 변형 데이터셋을 만들어 PARTIAL_DIR/dst_name 에 images/labels 구조로 저장
    # mode: 'crop' | 'occlude' | 'mixed'(이미지마다 crop/occlude 중 무작위)
    rng = np.random.default_rng(SEED)
    dst = PARTIAL_DIR / dst_name
    shutil.rmtree(dst, ignore_errors=True)         # 다시 실행하면 새로 생성 (시드 고정이라 결과 동일)
    (dst / 'images').mkdir(parents=True); (dst / 'labels').mkdir()
    imgs = list_images(src_dir)
    if sample_ratio < 1:
        imgs = sorted(random.Random(SEED).sample(imgs, int(len(imgs) * sample_ratio)))
    meta = []
    for p in imgs:
        img = cv2.imread(str(p)); H, W = img.shape[:2]
        boxes = read_labels(p, W, H)
        m = mode if mode != 'mixed' else ['crop', 'occlude'][rng.integers(2)]
        if m == 'crop':
            new_img, new_boxes, info = crop_edge(img, boxes, rng)
        else:
            r = ratio if ratio is not None else float(rng.choice(OCCLUDE_RATIOS))
            new_img, new_boxes = occlude(img, boxes, r, rng); info = f'occlude {r:.0%}'
        name = f'{p.stem}_{dst_name}'
        cv2.imwrite(str(dst / 'images' / f'{name}{p.suffix}'), new_img)
        write_yolo_labels(dst / 'labels' / f'{name}.txt', new_boxes, new_img.shape[1], new_img.shape[0])
        meta.append(dict(src=p.name, image=f'{name}{p.suffix}', mode=m, info=info,
                         boxes_before=len(boxes), boxes_after=len(new_boxes)))
    meta = pd.DataFrame(meta)
    print(f'[{dst_name}] {len(meta)}장 생성 | 라벨 박스 {meta.boxes_before.sum()} → {meta.boxes_after.sum()}개')
    return dst / 'images', meta

In [ ]:
# ── 1단계용 변형 test셋 생성 ──
TEST_VARIANTS = {'original': split_dirs['test']}      # 변형 이름 → 이미지 폴더
VARIANT_DST = {}                                      # 변형 이름 → 저장 폴더 이름 (미리보기에서 사용)
TEST_VARIANTS['crop'], crop_meta = make_variant(split_dirs['test'], 'test_crop', 'crop')
VARIANT_DST['crop'] = 'test_crop'
for r in OCCLUDE_RATIOS:
    key = f'occlude{int(r * 100)}'
    TEST_VARIANTS[key], _ = make_variant(split_dirs['test'], f'test_{key}', 'occlude', ratio=r)
    VARIANT_DST[key] = f'test_{key}'

display(crop_meta.head())   # 어떤 객체를 몇 % 잘랐는지 기록

### 라벨이 제대로 따라갔는지 눈으로 확인
원본과 변형 이미지에 **정답 박스**를 그려서 비교한다. 잘린/가려진 뒤에도 박스가 객체의 보이는 부분을 정확히 감싸고 있어야 한다.

In [ ]:
# 지게차와 사람이 함께 있는 test 이미지 3장을 골라 [원본, crop, occlude30, occlude50] 순서로 정답 박스 표시
preview = images_df[(images_df['split'] == 'test') & images_df['both']]['path'].head(3).tolist()
if not preview:   # 둘 다 있는 이미지가 없으면 아무 test 이미지 사용
    preview = images_df[images_df['split'] == 'test']['path'].head(3).tolist()

ims, titles = [], []
for p in preview:
    p = Path(p)
    ims.append(draw_gt(p)); titles.append('original (GT)')
    for key, dst_name in VARIANT_DST.items():
        vp = PARTIAL_DIR / dst_name / 'images' / f'{p.stem}_{dst_name}{p.suffix}'
        ims.append(draw_gt(vp)); titles.append(f'{key} (GT)')
show_grid(ims, titles, ncols=1 + len(VARIANT_DST), size=4, name='partial_label_check')

### ✍️ 라벨 점검 메모
- 박스가 보이는 부분을 잘 감싸는가? 이상한 사례가 있다면: 
- `MIN_VISIBLE`, `CUT_RANGE`, `OCCLUDE_RATIOS` 값이 현장 상황을 잘 반영하는가? 바꾼다면 왜: 

### 10-1. 1단계: 기존 모델로 부분 노출 test 평가 (재학습 없음)

In [ ]:
# 모든 test 변형에 대해 Before(사전학습) / After(추가 학습) 모델을 같은 함수로 평가
stage1 = []
for vname, vdir in TEST_VARIANTS.items():
    for model, cmap, mname in [(pretrained, PRETRAINED_MAP, 'Before'), (finetuned, FINETUNED_MAP, 'After')]:
        row, _ = evaluate(model, cmap, mname, img_dir=vdir)
        row['test_set'] = vname
        stage1.append(row)
stage1_df = pd.DataFrame(stage1)
stage1_df.to_csv(WORK_DIR / 'partial_stage1_results.csv', index=False)

show = ['model', 'test_set'] + [f'{CLASS_NAMES[c]}_{m}' for c in EVAL_CLASSES for m in ['R', 'AP50']]
display(stage1_df[show].round(3))

In [ ]:
def plot_by_testset(df, model_names, title, name):
    # 클래스별 Recall(윗줄)과 AP50(아랫줄)을 test 변형(x축)마다 모델별 막대로 비교
    order = list(TEST_VARIANTS)
    fig, axes = plt.subplots(2, len(EVAL_CLASSES), figsize=(7 * len(EVAL_CLASSES), 8))
    for j, c in enumerate(EVAL_CLASSES):
        for i, metric in enumerate(['R', 'AP50']):
            ax = axes[i, j]
            pv = df.pivot(index='test_set', columns='model', values=f'{CLASS_NAMES[c]}_{metric}').reindex(order)[model_names]
            pv.plot(kind='bar', ax=ax, rot=0)
            ax.set_title(f'{CLASS_NAMES[c]} - {"Recall" if metric == "R" else "AP50"}')
            ax.set_ylim(0, 1.05); ax.grid(alpha=0.3, axis='y'); ax.set_xlabel('')
    fig.suptitle(title); plt.tight_layout(); savefig(name); plt.show()

plot_by_testset(stage1_df, ['Before', 'After'], 'Stage 1: partial visibility (no retraining)', 'partial_stage1')

### ✍️ 1단계 해석
- 원본 대비 성능이 가장 많이 떨어진 변형과 클래스: 
- 사람과 지게차 중 부분 노출에 더 약한 쪽과 이유 추측: 
- 안전 관점에서 이 결과가 의미하는 것: 

### 10-2. 2단계: 부분 노출 이미지를 섞어서 재학습
- train 이미지의 `TRAIN_AUG_RATIO`(50%)로 crop/occlude 변형본을 만들어 **원본 train에 추가**한다. (원본을 대체하지 않음)
- val은 원본을 그대로 사용하고, 나머지 학습 조건(모델, epoch, 해상도)은 7장과 동일하게 맞춘다. → **데이터만 바꾼 비교**
- T4 기준 약 15~20분 추가 소요

In [ ]:
# train 변형본 생성 (이미지마다 crop 또는 occlude를 무작위 적용)
train_aug_dir, train_aug_meta = make_variant(split_dirs['train'], 'train_aug', 'mixed', sample_ratio=TRAIN_AUG_RATIO)
print(train_aug_meta['mode'].value_counts())   # crop/occlude가 각각 몇 장인지 확인

# train을 [원본 폴더, 변형 폴더] 두 개로 지정한 yaml (YOLO는 train에 폴더 목록을 받을 수 있음)
PARTIAL_YAML = PARTIAL_DIR / 'data_partial.yaml'
yaml.safe_dump({'path': str(DATA_ROOT),
                'train': [str(split_dirs['train']), str(train_aug_dir)],
                'val': str(split_dirs['val']), 'test': str(split_dirs['test']),
                'nc': len(CLASS_NAMES), 'names': CLASS_NAMES},
               open(PARTIAL_YAML, 'w'), allow_unicode=True, sort_keys=False)

# 7장과 같은 함수·조건으로 학습 (이름만 다르게)
partial_best = train_or_resume('finetune_partial', PARTIAL_YAML)
partial_model = YOLO(str(partial_best))

In [ ]:
# 원본 test + 모든 변형 test에서 [7장 모델(After)] vs [부분 노출 재학습 모델] 비교
stage2 = [r for r in stage1 if r['model'] == 'After']     # 7장 모델 결과는 1단계에서 이미 계산함
for vname, vdir in TEST_VARIANTS.items():
    row, _ = evaluate(partial_model, FINETUNED_MAP, 'After + partial', img_dir=vdir)
    row['test_set'] = vname
    stage2.append(row)
stage2_df = pd.DataFrame(stage2)
stage2_df.to_csv(WORK_DIR / 'partial_stage2_results.csv', index=False)

# 재학습으로 얼마나 변했는지 (양수 = 개선)
diff = (stage2_df[stage2_df.model == 'After + partial'].set_index('test_set')[show[2:]]
        - stage2_df[stage2_df.model == 'After'].set_index('test_set')[show[2:]])
print('재학습 후 변화량 (After + partial − After)')
display(diff.reindex(list(TEST_VARIANTS)).round(3))

plot_by_testset(stage2_df, ['After', 'After + partial'], 'Stage 2: retrained with partial-visibility images', 'partial_stage2')

### ✍️ 2단계 해석
- 변형 test에서 개선되었는가? 어느 변형/클래스에서 가장 효과가 컸는가: 
- **원본 test 성능은 유지되었는가?** (떨어졌다면 원인 추측): 
- YOLOv8 기본 증강(`erasing`, `mosaic`)에 이미 비슷한 효과가 있는데, 추가 효과가 있었는가: 
- 다음에 바꿔볼 조건 (예: `MIN_VISIBLE` 0.1로 낮추기, 가림 비율 70%, 실제 가려진 사진 수집): 

## 11. (확장) 거리 기반 경고 시연
최종 목표로 가기 위한 간단한 프로토타입이다.
- 카메라 보정 정보가 없어서 **사람 박스 높이를 평균 키 1.7m로 가정**해 픽셀→미터로 환산한다.
- 사람의 발 위치(박스 아래 중앙)에서 지게차 박스까지 최단 거리가 `DANGER_M` 보다 가까우면 **DANGER(안내 방송)** 로 판단한다.
- 한계: 원근, 앉아 있거나 가려진 사람, 지게차에 탄 운전자는 오차가 크다 → 실제로는 카메라 캘리브레이션이 필요하다.

In [ ]:
# 가정한 사람 키(m)와 경고 기준 거리(m): 3m 미만 DANGER, 6m 미만 CAUTION
PERSON_HEIGHT_M, DANGER_M, CAUTION_M = 1.7, 3.0, 6.0
LEVEL_COLOR = {'SAFE': (0, 200, 0), 'CAUTION': (0, 165, 255), 'DANGER': (0, 0, 255)}

def check_distance(result, conf_th=0.4):
    # 한 이미지의 예측 결과에서 모든 (사람, 지게차) 쌍의 추정 거리와 위험 단계를 계산
    # 오탐으로 인한 잘못된 경고를 줄이려고 신뢰도 0.4 이상인 박스만 사용
    d = result.boxes.data.cpu().numpy(); d = d[d[:, 4] >= conf_th]
    persons, forks = d[d[:, 5] == PERSON_ID][:, :4], d[d[:, 5] == FORKLIFT_ID][:, :4]
    pairs = []
    for p in persons:
        # 사람 박스 높이 = 1.7m 로 보고 1픽셀이 몇 m인지 계산
        m_per_px = PERSON_HEIGHT_M / max(p[3] - p[1], 1)
        # 사람이 바닥에 닿은 위치 = 박스 아래쪽 가운데
        foot = ((p[0] + p[2]) / 2, p[3])
        for f in forks:
            near = (min(max(foot[0], f[0]), f[2]), min(max(foot[1], f[1]), f[3]))   # 지게차 박스에서 가장 가까운 점
            dist = np.hypot(foot[0] - near[0], foot[1] - near[1]) * m_per_px
            level = 'DANGER' if dist < DANGER_M else 'CAUTION' if dist < CAUTION_M else 'SAFE'
            pairs.append((foot, near, dist, level))
    # 이미지 전체의 위험 단계 = 쌍들 중 가장 위험한 단계
    worst = max((pp[3] for pp in pairs), key=['SAFE', 'CAUTION', 'DANGER'].index, default='SAFE')
    return pairs, worst

# test 이미지마다 탐지 → 거리 계산 → 위험하면 '안내 방송' 메시지 출력 (실제 스피커 대신 print)
ims, titles, levels = [], [], []
for p in list_images(split_dirs['test']):
    r = finetuned.predict(str(p), device=DEVICE, verbose=False)[0]
    pairs, level = check_distance(r); levels.append(level)
    if pairs and len(ims) < 6:
        # 탐지 박스 위에 사람-지게차 연결선과 거리(m)를 위험 단계 색으로 표시
        img = r.plot()
        for foot, near, dist, lv in pairs:
            a, b = tuple(map(int, foot)), tuple(map(int, near))
            cv2.line(img, a, b, LEVEL_COLOR[lv], 3)
            cv2.putText(img, f'{dist:.1f}m', a, cv2.FONT_HERSHEY_SIMPLEX, 0.8, LEVEL_COLOR[lv], 2)
        if level == 'DANGER':
            print(f'🔊 [{p.name[:30]}] 안내 방송: 지게차 접근 주의!')
        ims.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); titles.append(level)

# 위험 단계별 이미지 수 집계
print(pd.Series(levels).value_counts())
if ims:
    show_grid(ims, titles, ncols=3, size=5, name='distance_alert_examples')

### ✍️ 경고 시연 관찰
- 거리 추정이 그럴듯했던 / 틀렸던 사례: 
- 실제 현장에 적용하려면 더 필요한 것: 

## 12. 결과 정리
| 구분 | 내용 |
|---|---|
| 문제 | 지게차–작업자 사각지대 접근 사고 |
| 방법 | 사전학습 YOLOv8n vs 지게차·사람 데이터로 fine-tuning한 YOLOv8n 비교 |
| 데이터 | Forklift-Person-Detector (Roboflow, 약 420장) |
| 핵심 결과 | ✍️ (예: forklift AP50 0.__ → 0.__ , person Recall 0.__ → 0.__) |
| 부분 노출 실험 | ✍️ (예: 50% 가림에서 person Recall 0.__ → 부분 노출 재학습 후 0.__) |
| 결론 | ✍️ |

## 13. 회고

### KPT
**Keep** — 이번에 잘한 점, 앞으로도 유지할 것
- ✍️

**Problem** — 어려웠던 점, 아쉬운 점
- ✍️ (예: Colab 런타임 초기화로 `data.yaml does not exist` 오류 → 원인을 찾는 데 시간이 걸림)

**Try** — 다음에 시도해 볼 것
- ✍️

### Action After Review (실행 가능한 액션)
| 액션 | 언제까지 | 완료 여부 |
|---|---|---|
| ✍️ (예: 야간/저조도 지게차 이미지 100장 추가 수집 후 재학습) | | ☐ |
| ✍️ (예: YOLOv8s와 속도·정확도 비교 실험) | | ☐ |
| ✍️ | | ☐ |

### 받은 피드백 기록
| 누구에게 | 피드백 내용 | 반영 여부 |
|---|---|---|
| ✍️ | | |

## 참고
- 데이터셋: luzuko, *Forklift-Person-Detector Dataset*, Roboflow Universe, CC BY 4.0 — https://universe.roboflow.com/luzuko/forklift-person-detector
- Ultralytics YOLOv8 문서 — https://docs.ultralytics.com